# Transfer Learning: Galaxy Classifier (full Galaxy10 DECaLS dataset)
Our task is to separate galaxy shapes (Galaxy10 DECaLS, 10 classes).
Adapted from the lesson's Animals starting code. This version uses the whole dataset from the .h5 file (80/20 train/validation split) and has no baseline.

In [ ]:
import torch
import torchvision

# Load VGG16 with the pre-trained weights.
vgg_model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)

# This standardization of R,G,B is specific to VGG. It expects 224x224 images (galaxies are 256x256).
proprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

: 

In [3]:
# Load dataset: the WHOLE Galaxy10 DECaLS dataset from the .h5 file, split 80% train / 20% validation.
# The full dataset as preprocessed float32 would be ~10.7 GB in RAM, so we keep the raw uint8 images
# (~3.5 GB) and run the same VGG preprocessing on each image as the DataLoader asks for it.
import os
import h5py
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

H5_PATH = os.path.join('..', 'data', 'Galaxy10_DECals.h5')
class_names = ['disturbed', 'merging', 'round_smooth', 'in_between_round_smooth', 'cigar_smooth',
               'barred_spiral', 'unbarred_tight_spiral', 'unbarred_loose_spiral',
               'edge_on_no_bulge', 'edge_on_with_bulge']  # label = index in this list (0-9)
num_classes = len(class_names)

with h5py.File(H5_PATH, 'r') as f:
    images = f['images'][:]                  # (17736, 256, 256, 3) uint8
    labels = f['ans'][:].astype(np.int64)    # (17736,)
print('Loaded', images.shape, labels.shape)

# Stratified so train and validation have the same class mix
train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.20, stratify=labels, random_state=42)
Y_train, Y_val = labels[train_idx], labels[val_idx]

class GalaxyDataset(Dataset):
    """Returns (image through the VGG preprocessing, label), shape (3, 224, 224) float32."""
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        img = Image.fromarray(images[j]).convert('RGB')
        return proprocess(img), int(labels[j])

for name, Y in [('train', Y_train), ('validation', Y_val)]:
    print(name, Y.shape, 'class counts:', np.bincount(Y, minlength=num_classes))

Loaded (17736, 256, 256, 3) (17736,)
train (14188,) class counts: [ 865 1482 2116 1622  267 1634 1463 2102 1139 1498]
validation (3548,) class counts: [216 371 529 405  67 409 366 526 284 375]


In [4]:
# Start by printing the model. Note that it has 3 parts: features, avgpool, and classifier.
# We'll do the simple thing, which is to remove the classifier.
print(vgg_model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [5]:
# This is a 10 class problem (the galaxy types), so we'll see if a single Linear layer works.
# We rebuild the model with a new classifier that has 10 outputs.
class GalaxyModel(torch.nn.Module):
    def __init__(self, vgg_model):
        super().__init__()
        self.features = vgg_model.features
        self.avgpool = vgg_model.avgpool
        self.classifier = torch.nn.Sequential(torch.nn.Linear(25088, num_classes))

    def forward(self, x):
        f = self.features(x)
        a = self.avgpool(f)
        flat = torch.flatten(a, 1)
        zE = self.classifier(flat)
        return zE

galaxy_model = GalaxyModel(vgg_model)
print(galaxy_model)

GalaxyModel(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, di

In [6]:
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [7]:
import time
import copy
import plotly.graph_objects as go
from torch.utils.data import DataLoader

# Will give the same random weights each time we run the code,
# which is helpful for debugging and reproducibility
torch.manual_seed(42)
model = GalaxyModel(copy.deepcopy(vgg_model)).to(device)

# Freeze the pretrained convolutional feature extractor.
# Only the new Linear classifier gets trained.
for param in model.features.parameters():
    param.requires_grad = False

# Hyperparameters
epsilon = 0.0001 # Learning rate.
batch_size = 32

# Shuffle the training data every epoch so batches differ from epoch to epoch.
# num_workers lets several CPU processes do the image preprocessing in parallel.
train_loader = DataLoader(GalaxyDataset(train_idx), batch_size=batch_size, shuffle=True, num_workers=4)

# Validation set is too big to push through VGG in one go, so evaluate in batches
val_loader = DataLoader(GalaxyDataset(val_idx), batch_size=batch_size, num_workers=4)

# Use CCE loss. Adam (adaptive momentum) is a fancy version of gradient descent
criterion = torch.nn.CrossEntropyLoss()
# Only pass the unfrozen parameters to the optimizer
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=epsilon)

/home/bittelbe/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [8]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 250890


In [9]:
def train_model(model, optimizer, num_epochs, checkpoint_path):
    """Train, keep the checkpoint with the lowest validation loss, and plot the losses."""
    training_losses = []
    validation_losses = []
    best_validation_loss = float('inf')
    best_validation_epoch = -1
    training_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        num_batches = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            optimizer.zero_grad()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        training_losses.append(avg_loss)

        # Compute validation loss. Don't need gradients here.
        model.eval()
        with torch.no_grad():
            val_total = 0
            for images, labels in val_loader:
                val_outputs = model(images.to(device))
                val_total += criterion(val_outputs, labels.to(device)).item() * len(labels)
            validation_loss = val_total / len(Y_val)
        validation_losses.append(validation_loss)

        # Save the model if this is the best validation loss so far
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_validation_epoch = epoch
            torch.save(model.state_dict(), checkpoint_path)
            best_marker = ' *'
        else:
            best_marker = ''

        print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {avg_loss:.5f}, Val Loss: {validation_loss:.5f}{best_marker}', flush=True)

    total_time = time.time() - training_start
    print(f'Average time per epoch: {total_time / num_epochs:.2f}s')
    print(f'Best validation loss: {best_validation_loss:.5f} (epoch {best_validation_epoch})')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(range(len(training_losses))), y=training_losses, mode='lines', name='Training Loss'))
    fig.add_trace(go.Scatter(x=list(range(len(validation_losses))), y=validation_losses, mode='lines', name='Validation Loss'))
    fig.update_layout(title="Training and Validation Loss Over Time", xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()

In [10]:
def evaluate_model(model, checkpoint_path):
    """Load the best checkpoint (lowest validation loss) and report validation accuracy and confusion matrix."""
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    predictions = []
    with torch.no_grad():
        for images, labels in val_loader:
            predictions.append(model(images.to(device)).argmax(dim=1).cpu())
    predictions = torch.cat(predictions).numpy()
    print(f'Validation accuracy: {(predictions == Y_val).mean():.4f}')

    # Confusion matrix: rows are the true class, columns are the predicted class
    confusion = np.zeros((num_classes, num_classes), dtype=int)
    for true, pred in zip(Y_val, predictions):
        confusion[true, pred] += 1
    print('Confusion matrix (rows = true, cols = predicted):', class_names)
    print(confusion)

## Transfer learning: pretrained VGG16 features (frozen) + new Linear classifier

In [11]:
train_model(model, optimizer, num_epochs=20, checkpoint_path='galaxy_classifier.pth')
evaluate_model(model, 'galaxy_classifier.pth')

/home/bittelbe/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [0/20], Train Loss: 1.46226, Val Loss: 1.24884 *
Epoch [1/20], Train Loss: 1.07411, Val Loss: 1.16591 *
Epoch [2/20], Train Loss: 0.92007, Val Loss: 1.11261 *
Epoch [3/20], Train Loss: 0.81058, Val Loss: 1.08594 *
Epoch [4/20], Train Loss: 0.72578, Val Loss: 1.07550 *
Epoch [5/20], Train Loss: 0.65879, Val Loss: 1.08445
Epoch [6/20], Train Loss: 0.59775, Val Loss: 1.08066
Epoch [7/20], Train Loss: 0.54724, Val Loss: 1.08570
Epoch [8/20], Train Loss: 0.49932, Val Loss: 1.07010 *
Epoch [9/20], Train Loss: 0.45894, Val Loss: 1.07510
Epoch [10/20], Train Loss: 0.42213, Val Loss: 1.08905
Epoch [11/20], Train Loss: 0.38950, Val Loss: 1.09373
Epoch [12/20], Train Loss: 0.35987, Val Loss: 1.10076
Epoch [13/20], Train Loss: 0.33343, Val Loss: 1.10712
Epoch [14/20], Train Loss: 0.30811, Val Loss: 1.12069
Epoch [15/20], Train Loss: 0.28343, Val Loss: 1.12064
Epoch [16/20], Train Loss: 0.26370, Val Loss: 1.14769
Epoch [17/20], Train Loss: 0.24364, Val Loss: 1.14438
Epoch [18/20], Train Loss:

/home/bittelbe/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Validation accuracy: 0.6406
Confusion matrix (rows = true, cols = predicted): ['disturbed', 'merging', 'round_smooth', 'in_between_round_smooth', 'cigar_smooth', 'barred_spiral', 'unbarred_tight_spiral', 'unbarred_loose_spiral', 'edge_on_no_bulge', 'edge_on_with_bulge']
[[ 34  24  26  26   3  12   6  72   6   7]
 [  2 223  44  27   1  11   6  35   2  20]
 [  2  16 454  23   0   1  16  16   0   1]
 [  7  30  91 260   1   0   3  13   0   0]
 [  2   1   1  11  26   1   0   5   2  18]
 [ 18  19   6  11   2 229  44  71   2   7]
 [  2  12  27   7   1  25 181 103   5   3]
 [ 22  38  27  18   4  48  35 316   9   9]
 [  5   3   0   0   0   0   0   6 234  36]
 [  2  12   4   8   1   2   1   3  26 316]]
